#### Environnement d'exécution :
MacBook Pro 2,6 GHz Intel Core i7 6 cœurs - 32 Go 2400 MHz DDR4

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
print(PROJECT_ROOT)

/Users/florianb/Downloads/ai-customer-insights-engine


In [2]:
import json
import time
from IPython.display import display, Markdown

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    ContextRelevance,
)
import pandas as pd

from src.rag import retriever as retriever_module
from src.rag import rag_chain as rag_chain_module

from config import config

In [3]:
import importlib

importlib.reload(config)
importlib.reload(retriever_module)
importlib.reload(rag_chain_module)

<module 'src.rag.rag_chain' from '/Users/florianb/Downloads/ai-customer-insights-engine/src/rag/rag_chain.py'>

In [4]:
print(f"HUGGINGFACE_EMBEDDING_MODEL = {config.HUGGINGFACE_EMBEDDING_MODEL}")

HUGGINGFACE_EMBEDDING_MODEL = sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [5]:
evaluation_path = PROJECT_ROOT / "data/evaluation/evaluation_questions.json"

with open(evaluation_path, "r", encoding="utf-8") as f:
    questions = json.load(f)

In [6]:
questions

['Quels types de problèmes rencontrent les clients avec le service client ?',
 "Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?",
 'Quelles difficultés les utilisateurs rencontrent-ils lors de la validation de leurs opérations ?',
 'Quels reproches sont faits concernant la réactivité du service client ?',
 'Comment les clients se sentent-ils face à des blocages de compte sans explication ?',
 'Quelles plaintes sont formulées concernant les frais bancaires ?',
 'Comment les clients perçoivent-ils la qualité des réponses fournies par le service client ?',
 'Quels problèmes sont signalés concernant la réception de cartes bancaires ?',
 'Comment les clients décrivent-ils leur expérience avec les délais de traitement des demandes ?',
 'Quelles frustrations sont exprimées concernant les communications par e-mail ?',
 'Quels avis négatifs sont partagés sur la gestion des incidents informatiques ?',
 'Comment les clients réagissent-ils face à des erreurs dan

In [7]:
# Retriever seul
retriever_only = retriever_module.create_retriever(
    model_name=config.HUGGINGFACE_EMBEDDING_MODEL,
    use_reranker=False,
    retriever_k=5,
)

# Retriever + reranker
retriever_reranker = retriever_module.create_retriever(
    model_name=config.HUGGINGFACE_EMBEDDING_MODEL,
    use_reranker=True,
    retriever_k=20,
    reranker_top_n=5,
)

### Context Relevance (Pertinence du Contexte)

Le LLM-as-a-judge évalue la pertinence des phrases / informations contenues dans les 5 contextes récupérés vis-à-vis de la question.

Formule : Nombre de phrases/informations jugées pertinentes par rapport à la question / Nombre total de phrases/informations dans les contextes récupérés

—> Score qui évalue la qualité du retrieval.

In [8]:
client = AsyncOpenAI(
    api_key=config.OPENAI_API_KEY
)

evaluation_llm = llm_factory(
    client=client,
    model="gpt-4o-mini",
    max_tokens=1200,
    temperature=0,
)

scorer = ContextRelevance(
    llm=evaluation_llm
)

In [9]:
# Résultats pour le retriever seul

# Chronomètre global
start_total = time.time()

results_retriever_only = []

for i, question in enumerate(questions, start=1):

    # Chronomètre du ainvoke()
    start_retrieval = time.time()

    docs = await retriever_only.ainvoke(question)

    retrieval_duration = time.time() - start_retrieval

    retrieved_contexts = [
        doc.page_content
        for doc in docs
    ]

    result = await scorer.ascore(
        user_input=question,
        retrieved_contexts=retrieved_contexts,
    )

    results_retriever_only.append({
        "question_id": i,
        "question": question,
        "retrieved_contexts": retrieved_contexts,
        "context_relevance": result.value,
        "retrieval_duration": retrieval_duration,
    })

total_duration = time.time() - start_total
minutes, seconds = divmod(int(total_duration), 60)

print(f"Durée totale d'exécution : {minutes} min {seconds} s")

Durée totale d'exécution : 0 min 34 s


In [10]:
# Résultats pour le retriever + reranker

# Chronomètre global
start_total = time.time()

results_retriever_reranker = []

for i, question in enumerate(questions, start=1):

    # Chronomètre du ainvoke()
    start_retrieval = time.time()

    docs = await retriever_reranker.ainvoke(question)

    retrieval_duration = time.time() - start_retrieval

    retrieved_contexts = [
        doc.page_content
        for doc in docs
    ]

    result = await scorer.ascore(
        user_input=question,
        retrieved_contexts=retrieved_contexts,
    )

    results_retriever_reranker.append({
        "question_id": i,
        "question": question,
        "retrieved_contexts": retrieved_contexts,
        "context_relevance": result.value,
        "retrieval_duration": retrieval_duration,
    })

total_duration = time.time() - start_total
minutes, seconds = divmod(int(total_duration), 60)

print(f"Durée totale d'exécution : {minutes} min {seconds} s")

Durée totale d'exécution : 3 min 38 s


In [11]:
comparison = []

for i in range(len(questions)):
    score_retriever_only = results_retriever_only[i]["context_relevance"]
    score_retriever_reranker = results_retriever_reranker[i]["context_relevance"]

    duration_retriever_only = results_retriever_only[i]["retrieval_duration"]
    duration_retriever_reranker = results_retriever_reranker[i]["retrieval_duration"]

    comparison.append({
        "n°": i+1,
        "Question": questions[i],

        "Contextes Retriever seul": "<br><br>".join(
            results_retriever_only[i]["retrieved_contexts"]
        ),
        "Score Retriever seul": score_retriever_only,
        "Durée Retriever seul": duration_retriever_only,

        "Contextes Retriever + Reranker": "<br><br>".join(
            results_retriever_reranker[i]["retrieved_contexts"]
        ),
        "Score Retriever + Reranker": score_retriever_reranker,
        "Durée Retriever + Reranker": duration_retriever_reranker,

        "Écart des scores": score_retriever_reranker - score_retriever_only,
        "Écart des durées": duration_retriever_reranker - duration_retriever_only,
    })

df_comparison = pd.DataFrame(comparison)

#df_comparison[["n°", "Question", "Score Retriever seul", "Score Retriever + Reranker", "Écart des scores"]].style.hide(axis="index")

In [12]:
# Moyennes des scores
mean_score_retriever_only = df_comparison["Score Retriever seul"].mean()
mean_score_retriever_reranker = df_comparison["Score Retriever + Reranker"].mean()

# Moyennes des durées
mean_duration_retriever_only = df_comparison["Durée Retriever seul"].mean()
mean_duration_retriever_reranker = df_comparison["Durée Retriever + Reranker"].mean()

df_comparison.loc[len(df_comparison)] = {
    "n°": "",
    "Question": "MOYENNE du Context Relevance (Pertinence du Contexte) / de la Durée",

    "Contextes Retriever seul": "",
    "Score Retriever seul": mean_score_retriever_only,
    "Durée Retriever seul": mean_duration_retriever_only,

    "Contextes Retriever + Reranker": "",
    "Score Retriever + Reranker": mean_score_retriever_reranker,
    "Durée Retriever + Reranker": mean_duration_retriever_reranker,

    "Écart des scores": mean_score_retriever_reranker - mean_score_retriever_only,
    "Écart des durées": mean_duration_retriever_reranker - mean_duration_retriever_only,
}

#df_comparison[["n°", "Question", "Score Retriever seul", "Score Retriever + Reranker", "Écart des scores"]].style.hide(axis="index")

In [13]:
def color_score(row):

    styles = pd.Series("", index=row.index)

    score_retriever_only = row["Score Retriever seul"]
    score_retriever_reranker = row["Score Retriever + Reranker"]

    # Ligne "MOYENNE"
    if row["Question"] == "MOYENNE du Context Relevance (Pertinence du Contexte) / de la Durée":
        styles[:] = "font-weight: bold; border-top: 2px solid black;"

    # Comparaison des scores
    if score_retriever_only > score_retriever_reranker:
        styles["Score Retriever seul"] += " background-color: lightgreen;"
        styles["Score Retriever + Reranker"] += " background-color: lightcoral;"

    elif score_retriever_only < score_retriever_reranker:
        styles["Score Retriever seul"] += " background-color: lightcoral;"
        styles["Score Retriever + Reranker"] += " background-color: lightgreen;"

    return styles

In [14]:
display(Markdown("### Comparaison du Context Relevance (Pertinence du Contexte)"))

df_comparison[
    [
        "n°",
        "Question",

        "Score Retriever seul",
        "Durée Retriever seul",

        "Score Retriever + Reranker",
        "Durée Retriever + Reranker",

        "Écart des scores",
        "Écart des durées"
    ]
].style \
    .hide(axis="index") \
    .format({
        "Score Retriever seul": "{:.2f}",
        "Score Retriever + Reranker": "{:.2f}",
        "Durée Retriever seul": "{:.2f} s",
        "Durée Retriever + Reranker": "{:.2f} s",
        "Écart des scores": "{:+.2f}",
        "Écart des durées": "{:+.2f} s",
    }) \
    .apply(color_score, axis=1) \
    .set_properties(
        subset=[
            "Score Retriever seul",
            "Score Retriever + Reranker",
            "Écart des scores",
        ],
        **{"border-left": "1px solid black"}
    )

### Comparaison du Context Relevance (Pertinence du Contexte)

n°,Question,Score Retriever seul,Durée Retriever seul,Score Retriever + Reranker,Durée Retriever + Reranker,Écart des scores,Écart des durées
1,Quels types de problèmes rencontrent les clients avec le service client ?,1.00,2.13 s,1.00,6.45 s,+0.00,+4.33 s
2,Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?,1.00,0.31 s,1.00,6.74 s,+0.00,+6.43 s
3,Quelles difficultés les utilisateurs rencontrent-ils lors de la validation de leurs opérations ?,1.00,0.21 s,1.00,7.77 s,+0.00,+7.56 s
4,Quels reproches sont faits concernant la réactivité du service client ?,1.00,0.23 s,0.75,7.18 s,-0.25,+6.95 s
5,Comment les clients se sentent-ils face à des blocages de compte sans explication ?,1.00,0.26 s,1.00,9.23 s,+0.00,+8.97 s
6,Quelles plaintes sont formulées concernant les frais bancaires ?,0.50,0.18 s,1.00,8.32 s,+0.50,+8.13 s
7,Comment les clients perçoivent-ils la qualité des réponses fournies par le service client ?,0.50,0.36 s,0.75,5.36 s,+0.25,+5.00 s
8,Quels problèmes sont signalés concernant la réception de cartes bancaires ?,1.00,0.08 s,1.00,10.12 s,+0.00,+10.04 s
9,Comment les clients décrivent-ils leur expérience avec les délais de traitement des demandes ?,0.75,0.22 s,1.00,13.28 s,+0.25,+13.06 s
10,Quelles frustrations sont exprimées concernant les communications par e-mail ?,1.00,0.34 s,1.00,18.46 s,+0.00,+18.12 s


In [15]:
pd.set_option("display.max_colwidth", None)

In [16]:
display(Markdown("### Comparaison du Context Relevance (Pertinence du Contexte)"))

df_comparison.style \
    .hide(axis="index") \
    .format({
        "Score Retriever seul": "{:.2f}",
        "Score Retriever + Reranker": "{:.2f}",
        "Durée Retriever seul": "{:.2f} s",
        "Durée Retriever + Reranker": "{:.2f} s",
        "Écart des scores": "{:+.2f}",
        "Écart des durées": "{:+.2f} s",
    }) \
    .apply(color_score, axis=1) \
    .set_properties(
        subset=[
            "Contextes Retriever seul",
            "Contextes Retriever + Reranker",
            "Écart des scores",
        ],
        **{"border-left": "1px solid black"}
    )

### Comparaison du Context Relevance (Pertinence du Contexte)

n°,Question,Contextes Retriever seul,Score Retriever seul,Durée Retriever seul,Contextes Retriever + Reranker,Score Retriever + Reranker,Durée Retriever + Reranker,Écart des scores,Écart des durées
1,Quels types de problèmes rencontrent les clients avec le service client ?,"Un service client incompétent - des processus incompréhensibles - même fermer son compte est compliquéservice client incompétant, raccroche au nezRéactivité du service clientService clients médiocre : incompétence, communication d’information partielle de manière intentionnelle, réponses contradictoires en fonction du conseiller en ligne, absence de réponse à vos messages…Un service client très compliqué et vraiment pas sympa",1.00,2.13 s,"Le service client est déplorable, le suivie d'un dossier sensible passe de main en main, sans qu'un service dédié prenne les choses aux sérieux afin de régler le problèmes. De plus ils refusent de vous communiqué autrement que par leur système de réclamation qui est non seulement difficile d'accès, fortement limité mais n'enregistre pas l'historique de votre dossier, ce qui complique encore la communication. Je déconseille fortement car au moindre soucis ça devient l'enfer.Le service client est pourri:Très bonne Banque, leur application est fluide, clair, puis simple à utiliserLe seul problème, c’est le service client c’est le point négatif. C’est le point noir très difficile de les contacter par téléphone ou par mail. Parfois il ne répond même pas aux e-mails. Et quand tu as l’opportunité d’avoir un conseiller par téléphone, ils sont froids et pas courtois.Service clients médiocre : incompétence, communication d’information partielle de manière intentionnelle, réponses contradictoires en fonction du conseiller en ligne, absence de réponse à vos messages…Client compte titre, service client inexistant lors de problèmes. Impossible de parler à quelqu'un qui ait un libre arbitre. Pas satisfait.Vu le peu de considération pour seulement bénéficier d'une offre commerciale existante, on peut craindre que ce service client ne soit encore moins aidant et conciliant sur des problèmes plus compliqués.",1.00,6.45 s,+0.00,+4.33 s
2,Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?,"Si je suis satisfait en tant que client. Mais je n'en dirai pas autant pour l'ouverture d'un compte. Ma mère voulait ouvrir un compte et le service client a été en dessous de tout on vous fixe un rdv pour un rappel 48h après pas de nouvelle.Super expérience d'ouverture de compte et de suivi. J'ai ouvert un compte joint chez Hellobank et n'était pas titulaire d'un compte chez eux avant, le niveau de clarté sur les démarches, suivi et réactivité de la part des conseillers en cas de besoin est vraiment exceptionnelle. Ça me donne confiance pour la suite!Excellente expérience de clôture d'un compte en alimentant le compte courant. Procédure immédiate et simple à comprendre. Comme pratiquement toujours chez Boursobank, satisfaction totale.Bonne expérience avec l'appli et la gestion du compte de mon adoL'ouverture d'un compte est une action que je trouve super bien encadrée les conseillers sont très avenants et parfaitement courtois. À l'inverse une fois le compte ouvert il est étrangement plus difficile d'obtenir un conseiller au téléphone et pour ma part j'attends depuis 3 semaines ma première carte bleue et il n'y a que très peu de réactions en face. Je trouve ça dommage mais tout ceci n'est que ma propre expérience et je souhaite aux autre utilisateurs de ne pas avoir les même soucis que moi !",1.00,0.31 s,"L'ouverture d'un compte est une action que je trouve super bien encadrée les conseillers sont très avenants et parfaitement courtois. À l'inverse une fois le compte ouvert il est étrangement plus difficile d'obtenir un conseiller au téléphone et pour ma part j'attends depuis 3 semaines ma première carte bleue et il n'y a que très peu de réactions en face. Je trouve ça dommage mais tout ceci n'est que ma propre expérience e